# CV data checks

The detector and recognizer share the saved page split and raw-pixel box coordinates. Validation is page-disjoint; writer/document identifiers are unavailable.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

from collections import Counter
import pandas as pd
from PIL import Image
from cv import CLASSES, pages_with_split
from htr import snapshot

pages = pages_with_split()
frame = pd.DataFrame([{'source': p['source'], 'split': p['split']} for p in pages])
pd.crosstab(frame.source, frame.split)

split,train,val
source,,
archive,114,13
dictation,323,36
school,614,68
university,146,16


In [2]:
assert all(region['type'] in CLASSES for page in pages for region in page['regions'])
assert all(0 <= r['bbox'][0] < r['bbox'][2] <= p['image_width']
           and 0 <= r['bbox'][1] < r['bbox'][3] <= p['image_height']
           for p in pages for r in p['regions'])
counts = {role: Counter(r['type'] for p in pages if p['split'] == role for r in p['regions'])
          for role in ('train', 'val')}
pd.DataFrame(counts).reindex(CLASSES).fillna(0).astype(int)

,train,val
handwritten,19470,2107
printed,304,4
formula,2630,262
table,132,10
annotation,486,68
image,111,8
graph,55,4


In [3]:
sample = [next(p for p in pages if p['source'] == source and p['split'] == 'train')
          for source in ('dictation', 'archive', 'university', 'school')]
checks = []
for page in sample:
    with Image.open(snapshot() / 'train' / page['file_name']) as image:
        assert image.size == (page['image_width'], page['image_height'])
        orientation = image.getexif().get(274, 1)
        assert orientation in (0, 1)
        checks.append(dict(source=page['source'], width=image.width, height=image.height, EXIF=orientation))
pd.DataFrame(checks)

,source,width,height,EXIF
0,dictation,3468,4624,1
1,archive,1352,2023,1
2,university,1669,2349,1
3,school,960,720,1


All metadata boxes are checked; pixel dimensions/orientation are spot-checked on one training page per source. The shared HTR crop CSV retains the existing targets and split.

![Dataset examples](../data/images/examples.png)